# Explorer le lakehouse depuis un notebook

Lecture seule, via le **pool analytique** (PostgreSQL + pg_lake) : toutes les tables du catalogue auxquelles le rôle
`analyste` a droit (gold, silver, bronze, données de l'application). Voir `docs/analytique.md`.

Avant de lancer :

```bash
kubectl -n spark port-forward svc/lake-analytique 5432:5432
pip install -r python/analyse/requirements.txt
export LAKE_MOT_DE_PASSE=…   # ANALYTIQUE_MOT_DE_PASSE de k8s/secrets/analytique.env
```

In [ ]:
import sys
sys.path.insert(0, "../python")          # le module python/analyse du dépôt
from analyse import tables, lire, exporter

## Ce que je peux lire

In [ ]:
tables()

## Lire en SQL, travailler en Python (pandas)

In [ ]:
ventes = lire("SELECT * FROM gold.ventes_par_jour WHERE date >= %s ORDER BY date", ("2026-11-01",))
ventes.describe()

In [ ]:
ventes.set_index("date")["chiffre_affaires"].astype(float).plot(title="Chiffre d'affaires par jour")

## Exporter un jeu de données

Le format suit l'extension : `.csv`, `.xlsx` (1 048 575 lignes au plus) ou `.parquet`. CSV et Parquet sont écrits
par lots, sans charger tout le résultat en mémoire.

In [ ]:
exporter("SELECT * FROM silver.ventes", "exports/ventes.parquet")
exporter("SELECT * FROM gold.ventes_par_jour", "exports/ventes_par_jour.xlsx")
exporter("SELECT * FROM vue.commandes WHERE statut = %s", "exports/commandes_ouvertes.csv", params=("ouverte",))